<a href="https://colab.research.google.com/github/jiyeonlee-2930/DeepLearning-TensorFlow-Basic/blob/main/8%EC%A3%BC%EC%B0%A8_%EA%B0%9D%EC%B2%B4%ED%83%90%EC%A7%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf  # tensorflow
import tensorflow_hub as tfhub  # tensorflow hub

In [ ]:
import os
import matplotlib.pylab as plt

# 로컬 이미지 파일 경로
local_img_path = '/content/myeongdong-1.jpg'

print(f"'{local_img_path}' 파일을 로드합니다.")
img = tf.io.read_file(local_img_path)
# JPEG 또는 PNG 등 다양한 이미지 형식을 처리하기 위해 decode_image를 사용합니다.
img = tf.image.decode_image(img, channels=3) # tf.image.decode_jpeg 대신 decode_image 사용
img = tf.image.convert_image_dtype(img, tf.float32) # 0 ~ 1 범위로 정규화

plt.figure(figsize=(15, 10))
plt.imshow(img)
plt.title(f"{os.path.basename(local_img_path)}")
plt.axis('off') # 축 정보 숨기기
plt.show()


In [ ]:
img_input = tf.expand_dims(img, 0)  # batch_size 추가
img_input.shape

In [ ]:
# TensorFlow Hub에서 모델 가져오기 - FasterRCNN+InceptionResNet V2
model = tfhub.load("https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1")

In [ ]:
# 모델 시그니처(용도, 서명) 확인
model.signatures.keys

In [ ]:
# 객체탐지 모델 생성
obj_detector = model.signatures['default']
obj_detector


In [ ]:
# 모델을 이용하여 예측 (추론)
result = obj_detector(img_input)
result.keys()

In [ ]:
# 탐지한 객체의 개수
len(result["detection_scores"])

In [ ]:
for key, value in result.items():
    print(key, value)

In [ ]:
# 객체 탐지 결과를 시각화
boxes = result["detection_boxes"]    # Bounding Box 좌표 예측값
labels = result["detection_class_entities"]   # 클래스 값
scores = result["detection_scores"]   # 신뢰도 (confidence)

# 샘플 이미지 가로 세로 크기
img_height, img_width = img.shape[0], img.shape[1]

# 탐지할 최대 객체의 수
obj_to_detect = 30

# 시각화
plt.figure(figsize=(15, 10))
for i in range(min(obj_to_detect, boxes.shape[0])):
    if scores[i] >= 0.2:
        (ymax, xmin, ymin, xmax) = (boxes[i][0]*img_height, boxes[i][1]*img_width,
                                    boxes[i][2]*img_height, boxes[i][3]*img_width)

        plt.imshow(img)
        plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin],
                 color='yellow', linewidth=2)

        class_name = labels[i].numpy().decode('utf-8')
        infer_score = int(scores[i].numpy()*100)
        annotation = "{}: {}%".format(class_name, infer_score)
        plt.text(xmin+10, ymax+20, annotation,
                 color='white', backgroundcolor='blue', fontsize=10)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np # numpy import 추가 (Colab 환경에 따라 필요할 수 있음)

# 객체 탐지 결과를 시각화
boxes = result["detection_boxes"]    # Bounding Box 좌표 예측값
labels = result["detection_class_entities"]   # 클래스 값
scores = result["detection_scores"]   # 신뢰도 (confidence)

# 샘플 이미지 가로 세로 크기
img_height, img_width = img.shape[0], img.shape[1]

# 탐지할 최대 객체의 수 (더 많은 객체를 보여주기 위해 증가)
obj_to_detect = 30

plt.figure(figsize=(15, 10))
plt.imshow(img) # 이미지를 한 번만 그림
current_axis = plt.gca() # 현재 축을 가져와서 사각형과 텍스트를 추가

print("--- Detected Objects ---")
# 신뢰도 점수가 높은 순으로 정렬하여 표시 (선택 사항이지만 좋은 관행)
sorted_indices = np.argsort(scores.numpy())[::-1]

for i in sorted_indices[:obj_to_detect]: # 상위 N개 객체만 고려
    score = scores[i].numpy()
    class_name = labels[i].numpy().decode('utf-8')

    # 신뢰도 점수가 0.5 이상인 경우에만 시각화
    if score >= 0.2:
        # Open Images 데이터셋의 바운딩 박스 좌표는 [ymin, xmin, ymax, xmax] 순서입니다.
        ymin, xmin, ymax, xmax = (boxes[i][0]*img_height, boxes[i][1]*img_width,
                                  boxes[i][2]*img_height, boxes[i][3]*img_width)

        # 'Woman' 또는 'Man' 객체에 특별히 관심을 가짐
        if class_name in ['Woman', 'Man']:
            color = 'cyan' # 여성/남성 감지 시 다른 색상
            print(f"Detected {class_name} with score: {int(score*100)}%")
        else:
            color = 'yellow'
            print(f"Detected {class_name} with score: {int(score*100)}%")

        # 바운딩 박스 그리기
        rect = plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                             fill=False, edgecolor=color, linewidth=2)
        current_axis.add_patch(rect)

        infer_score = int(score * 100)
        annotation = f"{class_name}: {infer_score}%"
        # 텍스트 위치를 박스 상단에 배치 (ymin - 10)
        plt.text(xmin, ymin - 10, annotation,
                 color='black', backgroundcolor=color, fontsize=10)

plt.title(f"Object Dectect Result: {os.path.basename(local_img_path)}")
plt.axis('off') # 축 정보 숨기기
plt.show()

# 얼굴 전용 모델로 교체


In [ ]:
import os
import matplotlib.pylab as plt

# 로컬 이미지 파일 경로
local_img2_path = '/content/people-1.jpg'

print(f"'{local_img2_path}' 파일을 로드합니다.")
img2 = tf.io.read_file(local_img2_path)
# JPEG 또는 PNG 등 다양한 이미지 형식을 처리하기 위해 decode_image를 사용합니다.
img2 = tf.image.decode_image(img2, channels=3) # tf.image.decode_jpeg 대신 decode_image 사용
img2 = tf.image.convert_image_dtype(img2, tf.float32) # 0 ~ 1 범위로 정규화

plt.figure(figsize=(15, 10))
plt.imshow(img2)
plt.title(f"{os.path.basename(local_img2_path)}")
plt.axis('off') # 축 정보 숨기기
plt.show()

In [ ]:
# ─────────────────────────────────────────
# 방법 1: OpenCV DNN 얼굴 검출 (가장 간단)
# ─────────────────────────────────────────
import cv2
import matplotlib.pyplot as plt
import numpy as np
import urllib.request

# 얼굴 검출 전용 모델 다운로드 (Caffe 기반)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
    "deploy.prototxt"
)
urllib.request.urlretrieve(
    "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20180205_fp16/res10_300x300_ssd_iter_140000_fp16.caffemodel",
    "face_detector.caffemodel"
)

# 모델 로드
net = cv2.dnn.readNetFromCaffe("deploy.prototxt", "face_detector.caffemodel")

# 이미지 로드
# img2는 현재 tf.Tensor (float32, 0-1, RGB) 입니다.
# OpenCV DNN 모델은 일반적으로 0-255 범위의 uint8 BGR 이미지를 입력으로 받습니다.

# 1. TensorFlow 텐서를 NumPy 배열로 변환하고, 0-255 범위의 uint8로 스케일링
image2_np_uint8 = (img2.numpy() * 255).astype(np.uint8)

# 2. RGB를 BGR로 변환 (OpenCV DNN 모델 입력용)
image2_bgr = cv2.cvtColor(image2_np_uint8, cv2.COLOR_RGB2BGR)

# 3. 시각화를 위해 원본 RGB 이미지 저장 (얼굴 검출 결과 그리기용)
image2_display_rgb = image2_np_uint8.copy() # 이 이미지는 나중에 plt.imshow에 사용됩니다.

# 얼굴 검출에 사용할 이미지와 높이, 너비 설정
# OpenCV DNN 모듈에는 BGR 이미지를 제공하는 것이 일반적입니다.
image2 = image2_bgr
h, w = image2.shape[:2]

# 얼굴 검출 실행
blob2 = cv2.dnn.blobFromImage(image2, 1.0, (300, 300), (104.0, 177.0, 123.0))
net.setInput(blob2)
detections = net.forward()

# ─────────────────────────────────────────
# 결과 시각화
# ─────────────────────────────────────────
# 결과 이미지는 원본 RGB 이미지에 얼굴 박스를 그릴 것이므로 image_display_rgb 사용
result2_img = image2_display_rgb.copy()
face_count = 0
conf_threshold = 0.2  # ← 더 낮은 임계값으로 더 많이 검출

print("--- Detected Faces ---")
for i in range(detections.shape[2]):
    confidence = detections[0, 0, i, 2]

    if confidence >= conf_threshold:
        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype("int")

        # 얼굴 박스 그리기
        cv2.rectangle(result2_img, (x1, y1), (x2, y2), (0, 255, 255), 2)

        label = f"Face: {int(confidence * 100)}%"
        cv2.rectangle(result2_img, (x1, y1-25), (x1+len(label)*11, y1),
                      (0, 255, 255), -1)
        cv2.putText(result2_img, label, (x1, y1-7),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        print(f"  [{face_count+1}] 신뢰도: {int(confidence*100)}%  좌표: ({x1},{y1}) → ({x2},{y2})")
        face_count += 1

plt.figure(figsize=(15, 10))
plt.imshow(result2_img)
plt.title(f"dnn_face_detector results : {face_count} people", fontsize=14)
plt.axis('off')
plt.show()
print(f"\n총 {face_count}명 검출됨")

# 얼굴 검출 + 성별 분류 모델 만들기

In [ ]:
# ─────────────────────────────────────────
# 방법 2: OpenCV DNN 얼굴 검출 (가장 간단) + 설병 분류
# ─────────────────────────────────────────
import cv2
import matplotlib.pyplot as plt
import numpy as np
import urllib.request

# 1. 얼굴 검출 전용 모델 다운로드 (Caffe 기반)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
    "deploy.prototxt"
)
urllib.request.urlretrieve(
    "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20180205_fp16/res10_300x300_ssd_iter_140000_fp16.caffemodel",
    "face_detector.caffemodel"
)
# 2. 성별 분류 모델 (Gil Levi & Tal Hassner 모델)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/GilLevi/AgeGenderDeepLearning/master/gender_net_definitions/deploy.prototxt",
    "gender_deploy.prototxt"
)
urllib.request.urlretrieve(
    "https://github.com/eveningglow/age-and-gender-classification/raw/master/model/gender_net.caffemodel",
    "gender_net.caffemodel"
)

# 모델 로드
face_net   = cv2.dnn.readNetFromCaffe("deploy.prototxt", "face_detector.caffemodel")
gender_net = cv2.dnn.readNetFromCaffe("gender_deploy.prototxt", "gender_net.caffemodel")

GENDER_LIST = ['Man', 'Woman']  # ← 원하는 클래스 라벨

# 이미지 로드
# img2는 현재 tf.Tensor (float32, 0-1, RGB) 입니다.
# OpenCV DNN 모델은 일반적으로 0-255 범위의 uint8 BGR 이미지를 입력으로 받습니다.

# 1. TensorFlow 텐서를 NumPy 배열로 변환하고, 0-255 범위의 uint8로 스케일링
image2_np_uint8 = (img2.numpy() * 255).astype(np.uint8)

# 2. RGB를 BGR로 변환 (OpenCV DNN 모델 입력용)
image2_bgr = cv2.cvtColor(image2_np_uint8, cv2.COLOR_RGB2BGR)

# 3. 시각화를 위해 원본 RGB 이미지 저장 (얼굴 검출 결과 그리기용)
image2_display_rgb = image2_np_uint8.copy() # 이 이미지는 나중에 plt.imshow에 사용됩니다.

# 얼굴 검출에 사용할 이미지와 높이, 너비 설정
# OpenCV DNN 모듈에는 BGR 이미지를 제공하는 것이 일반적입니다.
image2 = image2_bgr
h, w = image2.shape[:2]

# 얼굴 검출 실행
blob3 = cv2.dnn.blobFromImage(image2, 1.0, (300, 300), (104.0, 177.0, 123.0))
face_net.setInput(blob3)          # net → face_net
detections = face_net.forward()   # net → face_net

# 결과 이미지 초기화
result3_img = image2_display_rgb.copy()  # result3_img 로 통일
face_count = 0
conf_threshold = 0.2

GENDER_COLOR = {
    'Man'   : (100, 149, 237),
    'Woman' : (255, 105, 180),
}

# ─────────────────────────────────────────
# 결과 시각화
# ─────────────────────────────────────────
# 결과 이미지는 원본 RGB 이미지에 얼굴 박스를 그릴 것이므로 image_display_rgb 사용
result2_img = image2_display_rgb.copy()
print("--- Detected Faces ---")
for i in range(detections.shape[2]):
    confidence = detections[0, 0, i, 2]

    if confidence >= conf_threshold:
        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype("int")
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        face_crop = image2[y1:y2, x1:x2]

        if face_crop.size > 0:
            gender_blob = cv2.dnn.blobFromImage(
                face_crop, 1.0, (227, 227),
                (78.4263377603, 87.7689143744, 114.895847746),
                swapRB=False
            )
            gender_net.setInput(gender_blob)
            gender_preds = gender_net.forward()
            gender = GENDER_LIST[gender_preds[0].argmax()]
            gender_conf = int(gender_preds[0].max() * 100)
            color = GENDER_COLOR[gender]

            cv2.rectangle(result3_img, (x1, y1), (x2, y2), color, 2)
            label = f"{gender}: {int(confidence*100)}% (성별:{gender_conf}%)"
            cv2.rectangle(result3_img, (x1, y1-28), (x1+len(label)*10, y1),
                          color, -1)
            cv2.putText(result3_img, label, (x1, y1-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            print(f"  [{face_count+1}] {gender} | 얼굴:{int(confidence*100)}% | 성별:{gender_conf}%")
            face_count += 1

plt.figure(figsize=(15, 10))
plt.imshow(result3_img)
plt.title(f"dnn_face_detector results : {face_count} people", fontsize=14)
plt.axis('off')
plt.show()
print(f"\n총 {face_count}명 검출됨")

# 과제 : 본인의 사진이든 온라인 사진 등 탐지하고 싶은 사진을 업로드하여 객체탐지 실시

In [ ]:
import tensorflow as tf  # tensorflow
tf.keras.preprocessing.image.load_img('/content/students.jpg')

In [ ]:
import numpy as np

img2 = tf.keras.preprocessing.image.load_img('/content/students.jpg')

In [ ]:
img2 = tf.cast(img2,tf.float32) /255
img2_input = tf.expand_dims(img2, 0)  # batch_size 추가, 4차원 텐서로 입력
img2_input.shape

In [ ]:
# TensorFlow Hub에서 모델 가져오기 - FasterRCNN+InceptionResNet V2
import tensorflow_hub as tfhub
model = tfhub.load("https://tfhub.dev/google/faster_rcnn/openimages_v4/inception_resnet_v2/1")

In [ ]:
# 모델 시그니처(용도) 확인
model.signatures.keys()

In [ ]:
# 객체탐지 모델 생성
obj_detector = model.signatures['default']
obj_detector

In [ ]:
# 모델을 이용하여 예측 (추론)
result2 = obj_detector(img2_input)
result2.keys()

In [ ]:
for key, value in result2.items():
    print(key, value)

In [ ]:
labesls = result2["detection_class_labels"]
names = result2['detection_class_names']

In [ ]:
boxes = result2["detection_boxes"]    # Bounding Box 좌표 예측값
labels = result2["detection_class_entities"]   # 클래스 값
scores = result2["detection_scores"]   # 신뢰도 (confidence)

In [ ]:
# 탐지한 객체의 개수
len(result2["detection_scores"])

In [ ]:
boxes

In [ ]:
result2["detection_scores"]

In [ ]:
result2["detection_boxes"][0]

In [ ]:
result2["detection_class_entities"]

In [ ]:
import matplotlib.pylab as plt

# 객체 탐지 결과를 시각화
boxes = result2["detection_boxes"]    # Bounding Box 좌표 예측값
labels = result2["detection_class_entities"]   # 클래스 값
scores = result2["detection_scores"]   # 신뢰도 (confidence)

# 샘플 이미지 가로 세로 크기
img2_height, img2_width = img2.shape[0], img2.shape[1]

# 탐지할 최대 객체의 수
obj_to_detect = 30

# 시각화
plt.figure(figsize=(15, 10))
for i in range(min(obj_to_detect, boxes.shape[0])):
    if scores[i] >= 0.5:
        (ymax, xmin, ymin, xmax) = (boxes[i][0]*img2_height, boxes[i][1]*img2_width,
                                    boxes[i][2]*img2_height, boxes[i][3]*img2_width)

        plt.imshow(img2)
        plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin],
                 color='yellow', linewidth=2)

        class_name = labels[i].numpy().decode('utf-8')
        infer_score = int(scores[i].numpy()*100)
        annotation = "{}: {}%".format(class_name, infer_score)
        plt.text(xmin+10, ymax+20, annotation,
                 color='white', backgroundcolor='blue', fontsize=10)